In [1]:
import time
from typing import List
import numpy as np
import torch
import torch.nn as nn
import torchode
from pathlib import Path
import pickle

from scipy.integrate import solve_ivp as sp_solve_ivp
from ftnode.data import TrialsDataset
from tqdm.auto import tqdm

from sklearn.preprocessing import MinMaxScaler
from ftnode.utils import set_global_seed, _load_loop_wrapper
from ftnode.data import TrialsDataset
from ftnode.node import (
    FTNODE, FeluSigmoidMLP,FeluSigmoidMLPfeaturized,
     GeluSigmoidMLPfeaturized,
)

import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams['font.family']= 'serif'

device = 'cpu'
seed = 1234
set_global_seed(seed=seed)
random_state = 67

[Seed] Deterministic mode enabled (may reduce speed).


In [2]:
def hysteresis_ode(t,x,lam):
    return lam+x-x**3

In [3]:
n_lam = 51
n_traj = 51
lams = np.linspace(-1,1,n_lam)
xs = np.linspace(-2,2,n_traj)


In [4]:
t_max = 0.25
n_colloc = 101


Xs = []
Us = []
t = np.linspace(0,t_max,n_colloc)
for lami in tqdm(lams):
    for x0 in xs:
        sol = sp_solve_ivp(
            hysteresis_ode,
            t_span = [0,t_max],
            y0 = np.array(x0).reshape(-1),
            t_eval = np.linspace(0,t_max,n_colloc),
            args = (lami,)
        )

        Xs.append(sol.y.T)
        Us.append([lami])
Xs = np.array(Xs)
Us = np.array(Us)

  0%|          | 0/51 [00:00<?, ?it/s]

In [5]:
dXs = np.zeros_like(Xs)
T = t[np.newaxis,:,np.newaxis]
X_diff = Xs[:,2:,:] - Xs[:,:-2,:]
T_diff = T[:,2:,:] - T[:,:-2,:]

dXs[:,1:-1,:] = X_diff/T_diff
dXs[:,0,:] = (Xs[:,1,:] - Xs[:,0,:]) / (T[:,1,:] - T[:,0,:])
dXs[:,-1,:] = (Xs[:,-1,:] - Xs[:,-2,:]) / (T[:,-1,:] - T[:,-2,:])

In [6]:
scaler = MinMaxScaler(feature_range=(-1,1))
Xs_scaled = scaler.fit_transform(Xs.reshape(-1,1).reshape(-1,1)).reshape(-1,n_colloc,1)

In [7]:
dX_tensor = [
    torch.tensor(dxi,dtype=torch.float32,device=device) for dxi in dXs
]
X_tensor = [
    torch.tensor(xi,dtype=torch.float32,device=device) for xi in Xs_scaled
]
U_tensor = [
    torch.tensor(ui,dtype=torch.float32,device=device) for ui in Us
]

T_tensor = [
    torch.tensor(t,dtype=torch.float32, device=device) for _ in range(len(Xs))
]

In [8]:
class GradDataset(torch.utils.data.Dataset):
    def __init__(self, dX: List, X: List, T: List, U: List):
        self.dX = dX
        self.X = X
        self.T = T
        self.U = U
        # self.trans_idx = Transient_idx

    def __len__(self):
        return len(self.dX)

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError(
                f"Index {idx} is out of bounds of dataset size: {len(self)}."
            )

        dXi = self.dX[idx]
        Xi = self.X[idx]
        ti = self.T[idx]
        ui = self.U[idx]

        return dXi, Xi, ti, ui

dataset = GradDataset(dX = dX_tensor,X = X_tensor, T = T_tensor, U = U_tensor)

# Run $k$-Folds Cross-validation

In [9]:
k_folds = 10

In [10]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, SubsetRandomSampler
import time
import copy

In [11]:
avg_best_val_losses = []
avg_best_train_losses = []

In [12]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 200  
batch_size = 50 
learning_rate = 1e-2
print_every = 10
_precision = 6
random_state = random_state
solve_method = 'tsit5'


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)

val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(
        dims=[1,20,20, 1],
        activation=nn.SiLU(),
        lower_bound=-4,
        upper_bound=-0.1,
        init_type=None
    )


    g = GeluSigmoidMLPfeaturized(
        dims=[6, 20,20, 1],
        activation=nn.SiLU(),
        lower_bound=-2,
        upper_bound=2,
        freq_sample_step=1,
        feat_lower_bound=-1.5,
        feat_upper_bound=1.5,
        init_type=None
    )

    model = FTNODE(f, g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    fold_seed = random_state + fold
    set_global_seed(fold_seed)

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            x0i = Xi[:, 0, :]
            
            # ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            # u_func = lambda t: ui_expanded
            u_func = lambda t: ui
            func = lambda t, x: model_fold(t, x, u_func)

            opt.zero_grad()
            sol = torchode.solve_ivp(
                f=func,
                y0=x0i,
                t_eval=ti.squeeze(),
                method=solve_method,
            )
            
            Xi_pred = sol.ys


            # loss = loss_criteria(dXi, dXi_pred)
            loss = loss_criteria(Xi, Xi_pred)
                
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                x0i = Xi[:, 0, :]
                # ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                # u_func = lambda t: ui_expanded
                u_func = lambda t: ui
                func = lambda t, x: model_fold(t, x, u_func)

                sol = torchode.solve_ivp(
                    f=func,
                    y0=x0i,
                    t_eval=ti.squeeze(),
                    method=solve_method,
                )

                Xi_pred = sol.ys


                # loss = loss_criteria(dXi, dXi_pred)
                loss = loss_criteria(Xi, Xi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---
[Seed] Deterministic mode enabled (may reduce speed).


Fold 1:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 3.153360e-03, Val Loss = 8.882557e-04, Time = 3.026269e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 3.604665e-04, Val Loss = 1.553569e-04, Time = 3.309899e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 1.261076e-04, Val Loss = 8.228263e-05, Time = 3.610021e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 8.965355e-05, Val Loss = 6.812288e-05, Time = 3.276388e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 7.085828e-05, Val Loss = 8.721153e-05, Time = 3.502681e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 5.752934e-05, Val Loss = 4.518183e-05, Time = 3.677214e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 3.523497e-05, Val Loss = 2.822931e-05, Time = 3.748509e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 4.990214e-06, Val Loss = 3.797329e-06, Time = 3.127647e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 2.092690e-06, Val Loss = 2.815606e-06, Time = 3.069732e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 2.894867e-06, Val Loss = 5.277227e-06, Time = 2.954586e+00, lr = 1.000000e

Fold 2:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.568711e-03, Val Loss = 4.835104e-04, Time = 3.287847e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 2.717512e-04, Val Loss = 1.566084e-04, Time = 3.605910e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 1.178145e-04, Val Loss = 8.353593e-05, Time = 3.596452e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 8.045558e-05, Val Loss = 7.416339e-05, Time = 3.587830e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 7.098229e-05, Val Loss = 6.070737e-05, Time = 3.394603e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 5.385694e-05, Val Loss = 5.831339e-05, Time = 3.381676e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 3.374423e-05, Val Loss = 4.021310e-05, Time = 3.384988e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 8.665196e-06, Val Loss = 1.161410e-05, Time = 3.178831e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 3.158581e-06, Val Loss = 5.013297e-06, Time = 3.141753e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 1.929207e-06, Val Loss = 3.464359e-06, Time = 3.070302e+00, lr = 1.000000e

...

Fold 9:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.346929e-03, Val Loss = 5.354367e-04, Time = 3.024384e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 3.514759e-04, Val Loss = 2.076102e-04, Time = 3.469473e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 1.548731e-04, Val Loss = 1.213714e-04, Time = 3.421504e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 1.072672e-04, Val Loss = 7.580191e-05, Time = 3.427606e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 8.189948e-05, Val Loss = 7.532006e-05, Time = 3.476115e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 6.587963e-05, Val Loss = 6.533037e-05, Time = 3.618142e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 3.376758e-05, Val Loss = 2.527300e-05, Time = 3.408959e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 5.516008e-06, Val Loss = 4.896383e-06, Time = 3.112630e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 2.508229e-06, Val Loss = 2.302984e-06, Time = 2.970571e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 2.208290e-06, Val Loss = 1.764975e-06, Time = 2.967846e+00, lr = 1.000000e

Fold 10:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.242004e-03, Val Loss = 5.054940e-04, Time = 2.839813e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 3.174901e-04, Val Loss = 2.162101e-04, Time = 3.506789e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 1.623543e-04, Val Loss = 1.268539e-04, Time = 3.489307e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 9.147418e-05, Val Loss = 8.373509e-05, Time = 3.557262e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 6.298555e-05, Val Loss = 7.574069e-05, Time = 3.500809e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 4.912813e-05, Val Loss = 6.117255e-05, Time = 3.445190e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 1.415643e-05, Val Loss = 1.670386e-05, Time = 3.346566e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 5.325264e-06, Val Loss = 6.380226e-06, Time = 3.172875e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 2.639293e-06, Val Loss = 3.266399e-06, Time = 3.089495e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 9.842913e-06, Val Loss = 3.949204e-06, Time = 3.064447e+00, lr = 1.000000e

([9.064260556949459e-08], 0)